# Setup (Dependencies & Imports)

Install dependencies, clone the repo, and import required libraries and modules.

In [ ]:
# Install dependencies
!pip install -q transformers torch openai pillow torch scikit-learn sentence-transformers matplotlib seaborn

In [ ]:
# Clone the project repo
!git clone https://github.com/gizayceylan/FakeNews.git

# Add it to Python path
import sys
sys.path.append("/content/FakeNews")


In [ ]:
# Imports
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import torch
import torch.nn.functional as F
import time
import subprocess
import json

from tqdm import tqdm
from PIL import Image

# ML/DL Libraries
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import confusion_matrix, classification_report

# Import custom utility for OpenAI Client
from utils.api_key import load_openai_client


In [ ]:
# Set OpenAI API
client = load_openai_client()

# Metadata (Dataset Loading & Preparation)

Load the Fakeddit metadata, clean it, and select the subset of samples containing valid image URLs.

In [ ]:
# Unzip fakeddit_images.zip from /content/FakeNews/assets/content/master_pipeline_assets/
!unzip -q /content/FakeNews/assets/fakeddit_images.zip -d /content/FakeNews/assets

In [ ]:
# Load the pre-processed metadata
subset_path = "/content/FakeNews/assets/fakeddit_balanced_subset.csv"
image_df = pd.read_csv(subset_path)

# Define the path for the image directory (which was unzipped)
img_dir = "/content/FakeNews/assets/content/fakeddit_images"

# Update the DataFrame paths to reflect the new location
image_df['image_path'] = image_df['image_path'].apply(
    lambda p: os.path.join(img_dir, os.path.basename(p))
)

print(f"Loaded {len(image_df)} clean samples.")
print("Shape subset:", image_df.shape)
print(image_df["label"].value_counts())
image_df.head()


In [ ]:
# Test a sample
Image.open(image_df["image_path"].iloc[0])


In [ ]:
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Log GPU specs (model, driver, VRAM)
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        text=True
    ).strip()
    gpu_name, driver_version, vram_total = [x.strip() for x in smi.split(",")]
except Exception:
    gpu_name, driver_version, vram_total = "CPU/Unknown", None, None

# Text Similarity (SentenceTransformer Layer)

Compute semantic similarity between user-provided text and BLIP-generated captions using a pretrained SentenceTransformer model.

In [ ]:
# Load SentenceTransformer
sim_model = SentenceTransformer("all-mpnet-base-v2")
sim_model.to(device)

In [ ]:
# Get text similarity scores
def compute_similarity(text1, text2):
    emb1 = sim_model.encode(text1, convert_to_tensor=True, device=device)
    emb2 = sim_model.encode(text2, convert_to_tensor=True, device=device)
    sim = util.cos_sim(emb1, emb2).item()
    return float(sim)


In [ ]:
# Display an example
t1 = image_df["clean_title"].iloc[0]
t2 = image_df["blip2_caption"].iloc[0]
simt1t2 = compute_similarity(t1, t2)

print(f"User Text        : {t1}")
print(f"Generated Caption: {t2}")
print(f"Similarity Score : {simt1t2:.4f}")

# Classifier (RoBERTa Prediction Layer)

Run the pretrained RoBERTa model to obtain binary fake/real predictions with confidence scores.

In [ ]:
# Load RoBERTa model + tokenizer
model_name = "yaoyinnan/roberta-fakeddit"

roberta_tokenizer = AutoTokenizer.from_pretrained(model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(model_name)
roberta_model.to(device)
roberta_model.eval()

In [ ]:
# Classification helper
def classify_with_roberta(text):
    text = text.replace("\n", " ").strip()
    inputs = roberta_tokenizer(text, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = roberta_model(**inputs)
        probs = F.softmax(outputs.logits, dim=1).squeeze()

    pred = torch.argmax(probs).item()
    label = "Fake" if pred == 0 else "Real"
    confidence = probs[pred].item()
    return label, confidence


# LLM Reasoning (GPT-4.1-mini Layer)

Use an LLM to combine predictions from user text, BLIP caption, and semantic similarity to make a final decision.

In [ ]:
# LLM promt builder
def build_prompt(
        user_text, text_pred, text_conf,
        blip_caption, cap_pred, cap_conf,
        similarity_score):

    return f"""
You are a misinformation detection assistant.

Here is a news headline for an image: "{user_text}"
A text classifier predicted this text is: {text_pred}
With a confidence score of: {text_conf:.2f}

An AI-generated caption describes the same image as: "{blip_caption}"
A text classifier predicted this caption is: {cap_pred}
With a confidence score of: {cap_conf:.2f}

Cosine similarity between the headline and AI-generated caption is: {similarity_score:.2f}

Please:
1. Decide if the content is likely associated with real or fake news. Choose ONE of: Real / Fake
2. Provide a SHORT explanation (1–2 sentences).
3. Suggest ONE action the user can take.

Respond in this exact format:
- Decision:
- Explanation:
- Nudge:
"""


In [ ]:
# LLM call wrapper
def query_llm(prompt):
    resp = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt,
        max_output_tokens=150,
        temperature=0
    )
    return resp.output_text


In [ ]:
# Run pipeline
results = []

# Start timer
T0 = time.perf_counter()

for i, row in tqdm(image_df.iterrows(), total=len(image_df)): # use sample_df for N samples
    user_text = row["clean_title"]
    blip_caption = row["blip2_caption"]

    # RoBERTa on user caption
    text_pred, text_conf = classify_with_roberta(user_text)

    # RoBERTa on BLIP caption
    cap_pred, cap_conf = classify_with_roberta(blip_caption)

    # Similarity
    sim_score = compute_similarity(user_text, blip_caption)

    # LLM reasoning
    prompt = build_prompt(
        user_text, text_pred, text_conf,
        blip_caption, cap_pred, cap_conf,
        sim_score
    )
    llm_out = query_llm(prompt)

    results.append({
        "image_path": row["image_path"],
        "text": user_text,
        "caption": blip_caption,
        "true_label": row["label"],
        "text_pred": text_pred,
        "text_conf": text_conf,
        "caption_pred": cap_pred,
        "caption_conf": cap_conf,
        "similarity": sim_score,
        "llm_response": llm_out
    })

# Stop timer
T1 = time.perf_counter()
print(f"\nTotal runtime: {(T1 - T0)/60:.2f} min  ({T1 - T0:.1f} sec)")

# Save results
results_df = pd.DataFrame(results)
results_df.head()


In [ ]:
# LLM response parser
def parse_field(resp, key):
    m = re.search(rf"-\s*{key}\s*:\s*(.*)", resp, flags=re.IGNORECASE)
    return m.group(1).strip() if m else None

results_df["llm_decision"]    = results_df["llm_response"].apply(lambda r: parse_field(r, "Decision"))
results_df["llm_explanation"] = results_df["llm_response"].apply(lambda r: parse_field(r, "Explanation"))
results_df["llm_nudge"]       = results_df["llm_response"].apply(lambda r: parse_field(r, "Nudge"))


In [ ]:
# Display the final table with parsed llm response
results_df[[
    "text",
    "caption",
    "similarity",
    "true_label",
    "text_pred",
    "caption_pred",
    "llm_decision",
    "llm_explanation",
    "llm_nudge"
]]

# Evaluation (Metrics & Visualizations)

Compute accuracy, confusion matrices, and classification reports for classifiers and LLM outputs, and review similarity matrices.

In [ ]:
# Convert textual labels to ints
def label_to_int(x):
    if str(x).lower() == "fake":  return 0
    if str(x).lower() == "real":  return 1
    return None  # Unclear or invalid

# Make a new df for evaluation
eval_df = pd.DataFrame({
    "true_label": results_df["true_label"].astype(int),
    "text_label":  results_df["text_pred"].apply(label_to_int),
    "caption_label": results_df["caption_pred"].apply(label_to_int),
    "llm_label":  results_df["llm_decision"].apply(label_to_int),
})


In [ ]:
# Accuracies
text_acc = np.mean(eval_df["text_label"] == eval_df["true_label"])
cap_acc = np.mean(eval_df["caption_label"] == eval_df["true_label"])
llm_acc  = np.mean(eval_df["llm_label"] == eval_df["true_label"])

print("User text classifier accuracy:", round(text_acc, 3))
print("BLIP caption classifier accuracy:", round(cap_acc, 3))
print("LLM accuracy:", round(llm_acc, 3))


In [ ]:
# Confusion matrices

# User text classifier CM
text_cm = confusion_matrix(eval_df["true_label"], eval_df["text_label"])

# BLIP caption classifier CM
cap_cm = confusion_matrix(eval_df["true_label"], eval_df["caption_label"])

# LLM classifier CM
llm_cm = confusion_matrix(eval_df["true_label"], eval_df["llm_label"])

print("RoBERTa (Text) Confusion Matrix:\n", text_cm)
print("\nRoBERTa (Caption) Confusion Matrix:\n", cap_cm)
print("\nGPT (Combined) Confusion Matrix:\n", llm_cm)

In [ ]:
# Heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(text_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[0])
axes[0].set_title("RoBERTa (Text) Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(cap_cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[1])
axes[1].set_title("RoBERTa (Caption) Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

sns.heatmap(llm_cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Pred Fake", "Pred Real"],
            yticklabels=["True Fake", "True Real"],
            ax=axes[2])
axes[2].set_title("GPT (Combined) Confusion Matrix")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("True")

plt.tight_layout()
plt.show()

In [ ]:
# Classification reports
print("\n-----------------------------------------------------")
print("RoBERTa (Text) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["text_label"],
    target_names=["Fake", "Real"]
))

print("\n-----------------------------------------------------")
print("RoBERTa (Caption) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["caption_label"],
    target_names=["Fake", "Real"]
))

print("\n-----------------------------------------------------")
print("GPT (Combined) Classification Report:")
print("-----------------------------------------------------")
print(classification_report(
    eval_df["true_label"],
    eval_df["llm_label"],
    target_names=["Fake", "Real"]
))


# Log

In [ ]:
# Show runtime
runtime_sec = T1 - T0
print(f"Total runtime: {runtime_sec/60:.2f} min ({runtime_sec:.1f} sec)")

# Show GPU specs
display(gpu_name, driver_version, vram_total)

In [ ]:
# Log runtime and gpu specs
log = {
    "pipeline": "Image_to_Text_Fusion_ver2",
    "n_samples": len(image_df),
    "runtime_sec": runtime_sec,
    "gpu_name": gpu_name,
    "driver_version": driver_version,
    "vram_total": vram_total,
}
print(log)

# Save

In [ ]:
# Save logs
with open("I2TFv2_runtime_log.jsonl", "a") as f:
    f.write(json.dumps(log) + "\n")

In [ ]:
# Save results_df
results_df.to_csv("I2TFv2_results_df.csv", index=False)

# Save eval_df
eval_df.to_csv("I2TFv2_eval_df.csv", index=False)